# RE_403 · Quins tipus de municipis té Catalunya? · resolt

> **Repte resolt.** Una manera de resoldre'l, no l'única. Els números del text surten d'una execució real del quadern.

Treballes a l'Observatori del Territori. Hi ha 947 municipis, i cada any se'n publiquen desenes
d'indicadors: població, habitatge, renda, turisme, sòl, aigua, residus... Et demanen una
**tipologia** que una persona que no sap PCA pugui entendre.

**La pregunta:** quins tipus de municipis hi ha a Catalunya, què els distingeix, i on cau el teu?

L'enunciat complet, amb el diccionari de columnes, és a [RE_403](../RE_403_municipis_catalunya.ipynb). Si encara no l'has fet, fes-lo abans de llegir això.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Còpia congelada al repositori del curs (les dades originals són vives i canvien).
# Si tens el repositori clonat, llegeix la còpia local; si no (Colab), la baixa de GitHub.
DADES = "https://raw.githubusercontent.com/ZenidX/Machine-Learning/main/Machine%20Learning/04_reptes/dades/"

def llegeix(fitxer, **opcions):
    for ruta in ("dades/" + fitxer, "../dades/" + fitxer):
        if os.path.exists(ruta):
            return pd.read_csv(ruta, **opcions)
    return pd.read_csv(DADES + fitxer, **opcions)

df = llegeix("indicadors_municipis_2023_2024.csv")
print(df.shape)
df.head()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

## 1.1 · Una fila per municipi?

Quantes files hi hauria d'haver si fos una fila per municipi? Quantes n'hi ha? Per què? Mira
també les files que no tenen codi de municipi: què són? Queda't amb una taula amb **un sol any i
una fila per municipi**.

In [ ]:
print(df["any"].value_counts().to_dict())
print(df.loc[df["codi_municipi"].isna(), ["any", "municipi", "_01_pob_total"]])
df1 = df[(df["any"] == 2024) & df["codi_municipi"].notna()].reset_index(drop=True)
assert df1["codi_municipi"].is_unique and df1["municipi"].notna().all() and len(df1) == 947
print("files tretes:", len(df) - len(df1))

Hi ha **dos anys** (2023 i 2024) i, a cada any, una fila **sense codi ni nom** amb 8 milions
d'habitants: és el **total de Catalunya**, no un municipi. Si no es treu, és un punt enorme que
desfigura l'escalat i el PCA. Ens quedem amb el 2024: 947 municipis.

## 1.2 · Columnes que serveixen

Per a cada indicador `_NN_` numèric: quina proporció de nuls té, i quina de zeros? Un zero, té
sentit en aquella variable, o és un forat amb una altra disfressa? Mira també el rang: hi ha
alguna columna amb valors que no poden ser?

In [ ]:
ind = df1.filter(regex="^_").select_dtypes("number")
diag = pd.DataFrame({"nuls": ind.isna().mean(), "zeros": (ind == 0).mean(), "max": ind.max()}).round(3)
print((diag["nuls"] > 0.3).sum(), "indicadors amb més d'un 30 % de nuls")
diag.sort_values("zeros", ascending=False).head(12)

In [ ]:
print("lloguer = 0:", (df1["_12_hab_preu_mig_lloguer"] == 0).sum(), "municipis; població mediana d'aquests:",
      df1.loc[df1["_12_hab_preu_mig_lloguer"] == 0, "_01_pob_total"].median())
print("_29_comp_sol_suc_c, màxim:", df1["_29_comp_sol_suc_c"].max())

- Els indicadors d'habitatge `_13_` a `_16_` tenen més d'un 40 % de nuls (només municipis grans).
- `_12_hab_preu_mig_lloguer` val **0** a 393 municipis (petits): no és un lloguer gratuït, és un
  **nul disfressat** (no hi ha prou contractes per publicar-ne la mitjana).
- Molts indicadors de planejament (`_41_` a `_48_`, `_6x_`) són gairebé sempre 0: no discriminen.
- `_29_comp_sol_suc_c` té valors de l'ordre de 10¹⁴: no és una superfície plausible, no es fa servir.

## 1.3 · La teva selecció

Tria entre 8 i 15 indicadors per fer la tipologia. Pensa si són **absoluts** (habitants, places,
metres cúbics) o **relatius** (per habitant, per 100 habitants, percentatges), i justifica cada
tria en una línia. Decideix què fas amb els nuls que quedin.

In [ ]:
SEL = {
    "_02_pob_evolucio": "creixement de població (%)",
    "_03_pob_densitat": "densitat (hab/km²)",
    "_04_pob_td": "taxa de dependència",
    "_09_pob_etca_estacional": "població estacional",
    "_18_socioeco_toe": "taxa d'ocupació",
    "_19_socioeco_tar": "taxa d'atur registrat",
    "_20_socioeco_rnmp": "renda neta mitjana per persona",
    "_24_tur_places_100hab": "places turístiques per 100 hab",
    "_26_tur_places_huts_100hab": "places en HUT per 100 hab",
    "_33_comp_sol_suc_c_hab": "sòl urbà per habitant",
    "_56_ma_consum_aigua_hab": "aigua per habitant",
    "_59_ma_residus_hab_dia": "residus per habitant i dia",
    "_60_ma_residus_fr_selectiva": "% de recollida selectiva",
}
COLS = list(SEL)
M = df1[COLS].copy()
for c in ["_03_pob_densitat", "_56_ma_consum_aigua_hab", "_33_comp_sol_suc_c_hab", "_24_tur_places_100hab", "_26_tur_places_huts_100hab", "_09_pob_etca_estacional"]:
    M[c] = np.log1p(M[c].clip(lower=0))
print("nuls a la selecció:", M.isna().sum()[lambda s: s > 0].to_dict())
M = SimpleImputer(strategy="median").fit_transform(M)
assert M.shape == (947, 13) and np.isnan(M).sum() == 0

Tots **relatius** (per habitant, per 100 habitants, taxes): si hi poséssim la població, el PCA i el
k-means separarien primer Barcelona de la resta. Les sis variables molt asimètriques (densitat,
aigua, sòl, places turístiques, HUT, població estacional) passen per `log1p` abans d'escalar. Els pocs nuls de renda (20 municipis) s'imputen amb la mediana.

## 2.1 · El PCA

Estandarditza la selecció i fes el PCA. Quina variància expliquen les dues primeres components?
Quantes en calen per al 80 %?

In [ ]:
scaler = StandardScaler().fit(M); Z = scaler.transform(M)
pca = PCA().fit(Z)
acum = np.cumsum(pca.explained_variance_ratio_)
assert (np.diff(acum) > 0).all() and np.isclose(acum[-1], 1)
print("PC1, PC2:", pca.explained_variance_ratio_[:2].round(3), "| acumulada 2:", acum[1].round(3),
      "| components per al 80 %:", np.argmax(acum >= 0.8) + 1)

## 2.2 · Què vol dir cada eix

Mira els pesos de les dues primeres components. Quines columnes hi pesen més, i amb quin signe?
Posa un nom a cada eix amb paraules teves (per exemple, «de rural a urbà»). Busca els municipis
dels extrems de cada eix per comprovar que el nom té sentit.

In [ ]:
pesos = pd.DataFrame(pca.components_[:2].T, index=[SEL[c] for c in COLS], columns=["PC1", "PC2"]).round(2)
coords = pca.transform(Z)
print(pesos.sort_values("PC1"))
for j in (0, 1):
    o = df1.assign(pc=coords[:, j]).sort_values("pc")
    print(f"PC{j+1}  extrem -:", list(o["municipi"].head(5)), "\n      extrem +:", list(o["municipi"].tail(5)))

Les dues primeres components expliquen el 42 % (25 % + 16 %); per al 80 % en calen 7.

- **PC1, «de ciutat dormitori a poble de segona residència».** Negatiu: densitat i atur alts
  (Badia del Vallès, Salt, Santa Coloma de Gramenet, Ripollet). Positiu: places turístiques i HUT
  per habitant, residus, aigua i sòl urbà per habitant (Palau-sator, Pals, Naut Aran, el Port de la
  Selva).
- **PC2, «de turisme de masses a micropoble agrari».** Negatiu: població estacional, atur i residus
  (Santa Susanna, Lloret de Mar, Salou, Pineda de Mar). Positiu: taxa d'ocupació, dependència i
  recollida selectiva (Tavèrnoles, Bellprat, Malla).

El signe de cada component és arbitrari (pot sortir girat): el que compta és quines columnes van
juntes. Els noms dels extrems confirmen que els noms dels eixos tenen sentit.

## 2.3 · Projectar a mà

Calcula **a mà** les coordenades dels municipis a les components (amb `mean_` i `components_`) i
comprova que coincideixen amb `pca.transform`.

In [ ]:
a_ma = (Z - pca.mean_) @ pca.components_.T
assert np.allclose(a_ma, pca.transform(Z))
a_ma[:3, :2].round(3)

## 2.4 · Els tipus de municipis

Fes k-means (sobre les dades estandarditzades o sobre unes quantes components; decideix-ho i
digues per què). Tria k amb el colze, la silueta i les mides dels grups. Per a cada grup, dona el
perfil (mediana de cada indicador, en unitats originals) i tres municipis d'exemple.

In [ ]:
files = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z)
    files.append({"k": k, "inercia": km.inertia_, "silueta": silhouette_score(Z, km.labels_),
                  "mides": sorted(np.bincount(km.labels_).tolist())})
pd.DataFrame(files).set_index("k").round(3)

In [ ]:
K = 4
km = KMeans(n_clusters=K, n_init=10, random_state=0).fit(Z)
assert np.bincount(km.labels_).sum() == len(df1)
perfil = df1.groupby(km.labels_)[COLS].median().rename(columns=SEL)
perfil["municipis"] = np.bincount(km.labels_)
perfil["població mediana"] = df1.groupby(km.labels_)["_01_pob_total"].median()
exemples = df1.groupby(km.labels_)["municipi"].apply(lambda s: list(s.sample(min(3, len(s)), random_state=0)))
print(exemples)
perfil.T.round(1)

La silueta és baixa per a totes les k (màxim 0,19 a k = 3): els municipis formen un continu, no
grups nets. k = 3 és la de silueta més alta; k = 4 (0,15) separa a més els micropobles dels pobles
rurals, sense grups minúsculs (el més petit en té 94). Amb k = 4 surten quatre tipus:

- **micropobles en declivi** (densitat de 8 hab/km², població que baixa, poc atur);
- **pobles rurals** (densitat d'uns 16 hab/km², població estable);
- **municipis turístics** (població estacional i places turístiques altíssimes: Vielha, Palafrugell,
  Castelló d'Empúries);
- **municipis urbans i metropolitans** (densitat d'uns 200 hab/km², creixement, atur més alt).

Els números dels grups (0-3) els posa k-means a l'atzar; la descripció surt del perfil.

## 2.5 · I el teu?

On cau el teu municipi (o el de l'institut)? A quin grup és, quins municipis té més a prop al
mapa de les dues primeres components, i et sembla que el retrat és just?

In [ ]:
i = df1.index[df1["municipi"].str.contains("Sant Cugat del Vall")][0]
urba = df1.groupby(km.labels_)["_03_pob_densitat"].median().idxmax()
assert km.labels_[i] == urba
print(df1.loc[i, "municipi"], "-> grup", km.labels_[i], "(el de densitat més alta)")
d2 = np.linalg.norm(coords[:, :2] - coords[i, :2], axis=1)
dz = np.linalg.norm(Z - Z[i], axis=1)
print("veïns al pla PC1-PC2:   ", list(df1.loc[np.argsort(d2)[1:7], "municipi"]))
print("veïns amb tots els indicadors:", list(df1.loc[np.argsort(dz)[1:7], "municipi"]))

Sant Cugat del Vallès (on és l'institut) cau al grup **urbà**. Però els seus veïns al pla de les dues
primeres components són municipis com l'Espluga de Francolí o Ponts, que no s'hi assemblen
gaire: aquest pla només guarda el 42 % de la variància, i el que fa diferent Sant Cugat (renda
alta, ocupació) hi queda aplanat. Buscant veïns amb **tots** els indicadors estandarditzats, la
llista s'assembla molt més al que esperaria qualsevol persona de la zona (Sant Just Desvern,
Cerdanyola del Vallès, Cabrera de Mar, Sant Vicenç de Montalt...). Lliçó: el mapa de 2D és
per mirar, no per mesurar. Fes-ho amb el teu municipi.

## Conclusió

Amb 13 indicadors relatius del 2024 (947 municipis, sense la fila del total de Catalunya), els
municipis formen un continu més que no pas grups nets (silueta màxima 0,19). Les dues primeres
components expliquen el 42 %: la primera va de ciutat dormitori densa a poble de segona residència,
i la segona, de turisme de masses a micropoble agrari. Amb k = 4 surten quatre tipus: micropobles en
declivi, pobles rurals, municipis turístics i municipis urbans i metropolitans. Sant Cugat del Vallès
cau al grup urbà, però els seus veïns al mapa de dues components no s'hi assemblen; amb tots els
indicadors, sí. El mapa de dues components és per mirar, no per mesurar.